# RAG: Semantic Clustering

This notebook clusters an existing vector database collection by embedding similarity,
labels each cluster with an LLM, and indexes cluster summaries as a navigational layer.


**IMPORTANT**: Run `example_RAG_01_load.ipynb` first to populate the 'books' collection.

## Initialize

In [ ]:
from agentic_patterns.agents.rag.clustering import label_clusters
from agentic_patterns.core.vectordb import get_vector_db
from agentic_patterns.core.vectordb.clustering import cluster
from agentic_patterns.core.vectordb.models import (
    Chunk,
    ChunkLevel,
    ClusterAlgorithm,
    DimReducer,
)

In [ ]:
vdb = get_vector_db("books")
count = vdb.count()
assert count > 0, "Vector database is empty. Run example_RAG_01_load.ipynb first."
print(f"Collection has {count} documents")

## Clustering

### Cluster with UMAP + HDBSCAN

UMAP reduces embedding dimensions to a space where density-based clustering works reliably.
HDBSCAN then discovers the number of clusters automatically, labeling outliers as noise (-1).

In [ ]:
result = cluster(
    vdb, algorithm=ClusterAlgorithm.HDBSCAN, reduce_dim=DimReducer.UMAP, n_components=10
)

print(f"Found {len(result.clusters)} clusters")
for c in result.clusters:
    label = "(noise)" if c.cluster_id == -1 else f"cluster {c.cluster_id}"
    print(f"  {label}: {len(c.items)} items")

In [ ]:
labeled_umap = await label_clusters(result)

for c in labeled_umap.clusters:
    if c.cluster_id == -1:
        print(f"  (noise): {len(c.items)} unclustered items")
        continue
    print(f"\nCluster {c.cluster_id}: {c.label}")
    print(f"  Summary: {c.summary}")
    print(f"  Size: {len(c.items)} items")
    print(f"  Sample: {c.items[0].text[:120]}...")

### Cluster with Spherical K-Means

Spherical k-means normalizes embeddings to the unit sphere before clustering,
making it equivalent to minimizing cosine distance. Use when the number of topics is known.

In [ ]:
result_skm = cluster(vdb, algorithm=ClusterAlgorithm.SPHERICAL_KMEANS, n_clusters=8)
labeled = await label_clusters(result_skm)

print("Spherical K-Means clusters (k=8):")
for c in labeled.clusters:
    print(f"  [{c.cluster_id}] {c.label} ({len(c.items)} items)")

### Cluster with Regular K-Means

Standard k-means uses Euclidean distance. Less appropriate for embeddings than spherical
k-means, but included for comparison.

In [ ]:
result_km = cluster(vdb, algorithm=ClusterAlgorithm.KMEANS, n_clusters=8)
labeled_km = await label_clusters(result_km)

print("K-Means clusters (k=8):")
for c in labeled_km.clusters:
    print(f"  [{c.cluster_id}] {c.label} ({len(c.items)} items)")

## Two-stage index & retrieval

### Index cluster summaries as a navigational layer

Storing cluster labels and summaries in a separate collection enables
two-stage retrieval: find the relevant cluster, then retrieve within it.

In [ ]:
vdb_index = get_vector_db("books_cluster_index")

index_chunks = []
for c in labeled.clusters:
    # Skip noise cluster - in practice, you might want to handle these separately
    if c.cluster_id == -1:
        continue
    doc_ids = ",".join(item.doc_id for item in c.items)
    index_chunks.append(
        Chunk(
            doc_id=f"cluster-{c.cluster_id}",
            text=f"{c.label}: {c.summary}",
            level=ChunkLevel.DOCUMENT,
            parent_id=None,
            metadata={
                "cluster_id": c.cluster_id,
                "item_count": len(c.items),
                "doc_ids": doc_ids,
            },
        )
    )

added = vdb_index.ingest(index_chunks, force=True)
print(f"Indexed {added} cluster summaries into 'books_cluster_index'")

### Two-stage retrieval via cluster index

First find the relevant cluster, then retrieve chunks from within that cluster only.

In [ ]:
query = "Characters facing moral dilemmas"

# Stage 1: find the best matching cluster
cluster_hits = vdb_index.retrieve(query=query, max_results=2)
print("Matching clusters:")
for hit in cluster_hits:
    # Scores are negative cosine distances: closer to 0 means more similar
    print(f"  score={hit.score:.3f} | {hit.text.replace(chr(10), ' ')}")

# Stage 2: retrieve within the best cluster's doc_ids
if cluster_hits:
    best_cluster_meta = cluster_hits[0].metadata
    doc_id_list = best_cluster_meta.get("doc_ids", "").split(",")
    print(f"\nRetrieving from {len(doc_id_list)} chunks in this cluster")

    cluster_docs = vdb.collection.get(
        ids=doc_id_list[:20], include=["documents", "metadatas"]
    )
    for doc_id, doc in zip(cluster_docs["ids"][:3], cluster_docs["documents"][:3]):
        print(f"  [{doc_id}] {doc[:100].replace(chr(10), ' ')}...")